# CM02 - Laplacian-smoothed grain boundary interfaces

Extract internal grain boundary surfaces directly from the voxel labels, smooth them with Laplacian relaxation, and display only those surfaces. No tetrahedra are generated. RVE exterior caps are omitted from this view.

Surface nodes smooth along surface adjacency; junction-line nodes use junction neighbors. Junction endpoints, branches and four-or-more-grain points stay frozen. RVE-plane coordinates remain fixed. This preview has triangle-collapse safeguards, but no volume-element quality constraints; its surface geometry can differ from CM01. It does not certify absence of self-intersections.

Run from inside this checkout with NumPy and PyVista installed in the notebook kernel.

## Control guide

These notes describe the existing controls. After changing an upstream setting, rerun its cell and all dependent cells below it. Assignments such as `CAP_MESH_SIZE = GMSH_SIZE` take effect when that cell runs; they are not dynamic links. Physical lengths use the units of `SPACING`. Increasing a limit helps only if that limit is actually preventing progress.

### Input and voxel cleaning

| Control | Meaning; effect of increasing or decreasing |
|---|---|
| `SAMPLE`, `ROOT` | Input data and repository paths. Changing the sample changes the structure. |
| `GRAIN_ID_SEED` | Reproducible random ID/colour permutation. Different integers give different permutations; larger is not necessarily more contrasting. Geometry is unchanged. |
| `CLEAN_SPIKES` | `True` enables UPXO spike removal before refinement; `False` bypasses it. |
| `SPIKE_MAX_PASSES` | Higher allows more cleaning passes; lower stops sooner. No-change and repeated-configuration stops still apply. More passes cannot override protected last voxels. |
| `CLEAN_PINCHES` | Enables targeted voxel contact cleaning; `False` bypasses it. |
| `PINCH_MAX_PASSES` | Higher allows more search passes; lower limits work. Other budgets and acceptance rules still apply. |
| `PINCH_MAX_CHANGES` | Higher allows more accepted voxel reassignments in total; lower is more conservative. |
| `PINCH_MAX_VOLUME_FRACTION` | Higher permits larger per-grain net voxel-count changes; lower preserves counts more tightly. Positive values allow at least one voxel; zero allows none. This is not a displacement limit or total-edit fraction. |
| `PINCH_PROTECT_GRAINS_UP_TO` | Higher protects more small grains from donating voxels; lower permits cleaning smaller grains. Counts are measured after spike cleaning, before upscaling. |
| `UPSCALE_FACTOR` | Integer ≥1: 1 leaves resolution unchanged; higher subdivides along all axes. Memory grows approximately with factor³. Refinement alone preserves the original geometry and contacts; it adds no physical detail. |
| `SPACING` | Original voxel dimensions x/y/z. Increasing a component enlarges that physical dimension; decreasing shrinks it. Use measured spacing, not this setting as a repair control. |

`effective_spacing` is derived from `SPACING / UPSCALE_FACTOR`. Settings using `min(SPACING)` remain tied to original physical voxel size. The pinch detector searches local 2×2×2 face-disconnected arrangements. Its budgets cannot extend detection to every thin neck or later smoothing-induced defect. Accepted edits preserve global six-connected component counts and obey volume/protection rules.

### Surface and junction smoothing

| Control | Increase / decrease effect |
|---|---|
| `LAPLACE_ITERATIONS` | More surface-smoothing steps can reduce staircasing but increase shrinkage, distortion and runtime; fewer retain more voxel detail. Zero disables these steps. |
| `LAPLACE_RELAXATION` | Larger values give stronger averaging steps; smaller are gentler. Use the supported range (0,1]. Local safeguards can suppress movement. |
| `JUNCTION_ITERATIONS` | More steps smooth eligible junction lines further; fewer retain more original shape. Zero disables line smoothing. |
| `JUNCTION_RELAXATION` | Larger values give stronger line-smoothing steps; smaller are gentler. Range (0,1]. |
| `JUNCTION_MAX_DISPLACEMENT` | Larger physical bound permits more total junction movement from extracted geometry; smaller preserves shape more tightly. Extra iterations may do little once constrained. |

Junction endpoints, branches and four-or-more-grain nodes remain frozen, and RVE-plane coordinates stay fixed. Stronger smoothing does not remove these topology constraints and can bring neighbouring sheets closer. Local collapse/reversal safeguards do not certify freedom from global intersections.

### Pre-Gmsh point-contact separation

| Control | Increase / decrease or switch effect |
|---|---|
| `SEPARATE_PINCHES` | Enables splitting disconnected triangle fans at a shared vertex; `False` bypasses it. |
| `PINCH_PRESS_RADIUS` | Larger physical radius affects a broader neighbourhood; smaller localizes pressing near the split tip. |
| `PINCH_PRESS_MAX_DISPLACEMENT` | Larger allows more movement; smaller better preserves shape but may prevent separation. |
| `PINCH_PRESS_ITERATIONS` | More steps permit further adjustment; fewer reduce work and deformation. Constraints still apply. |
| `PINCH_PRESS_RELAXATION` | Larger gives stronger steps; smaller gives gentler adjustment. |
| `PINCH_MIN_OPENING` | Larger demands a wider minimum separation of duplicated tips and can cause rollback if unattainable; smaller accepts narrower separation. It is not a global clearance guarantee. |

Length controls use physical units. This operation handles disconnected point fans: it does **not** cut shared edges or connected multi-grain junctions. Increasing its controls cannot remove contacts outside that scope. Other frozen junction points and RVE coordinates remain constrained.

### Gmsh and RVE caps

| Control | Increase / decrease effect |
|---|---|
| `GMSH_SIZE` | Smaller requests finer surface triangles and greater cost; larger requests coarser triangles. Retained curves and chart seams constrain size. Refinement does not itself remove contacts or overlaps. |
| `GMSH_ALGORITHM` | Categorical: 1 MeshAdapt, 2 Automatic, 5 Delaunay, 6 Frontal-Delaunay. Higher numbers are not inherently better. |
| `CAP_MESH_SIZE` | Smaller requests finer new cap triangles; larger requests coarser ones. |
| `RVE_EDGE_SIZE` | Smaller subdivides new box-edge segments more finely; larger permits longer segments. Existing interface traces remain constrained. |
| `FINAL_GMSH_SIZE` | Same size tradeoff, applied to joint remeshing of internal interfaces and caps. |
| `FINAL_GMSH_ALGORITHM` | Same categorical choices as `GMSH_ALGORITHM`. |

The reported “internal triangle is coplanar with the cap” error is a geometry/classification check. Changing cap size alone does not address that condition. Its investigation and repair remain deferred until **123go**.

### Validation, later repairs and tetrahedra

| Control | Increase / decrease or switch effect |
|---|---|
| `MIN_SURFACE_TRIANGLE_QUALITY` | Higher flags more poor triangles; lower accepts more. This is a checking threshold, not an improvement operation. |
| `STOP_ON_VALIDATION_ERRORS` | `True` raises after saving the report; `False` permits later cells but does not make failed geometry valid. |
| `REPAIR_CONTACTS` | Enables the later contact repair; `False` bypasses it. |
| `REPAIR_SEPARATION` | Larger physical movement limit permits stronger separation; smaller is more conservative. Only supported ambiguous edges/fans are handled; genuine multi-grain junction edges remain shared. |
| `REPAIR_INTERFACE_CROSSINGS` | Enables the supported local crossing repair; `False` bypasses it. |
| `CROSSING_MAX_DISPLACEMENT` | Larger permits larger physical corrections; smaller can reject required movement. |
| `CROSSING_NEIGHBOURHOOD_RADIUS` | Larger uses a broader region to estimate separation directions; smaller uses more local normals. It is not a general collision-search distance. |
| `CROSSING_OPENING` | Larger dimensionless value requests wider wedge separation relative to distance from the junction line; smaller is gentler. Excessive opening can violate movement or inversion checks. |
| `RUN_TET_VERIFICATION` | Enables actual tetrahedral generation and checks. Disabling it does not establish readiness. |
| `TET_MESH_SIZE` | Smaller requests finer volume elements at greater cost; larger requests coarser ones, subject to surface constraints. Neither direction guarantees better minimum quality. |
| `TET_OPTIMIZE_NETGEN` | Enables optimization, retaining the baseline for a volume when its minimum quality worsens. |
| `MIN_TET_QUALITY` | Higher makes acceptance stricter; lower admits worse tetrahedra. It does not itself improve elements. |

Crossing repair supports particular interface wedges beside shared junction lines, not arbitrary overlaps or coplanar triangles. Revalidation is required after repairs.

### Display and output controls

| Control | Effect |
|---|---|
| `GRAIN_ID` | `None` shows all interfaces; a shuffled ID selects that grain's interfaces. |
| `AUDIT_ORIGINAL_GRAIN_ID` | Original unshuffled ID to trace; `None` selects a smallest input grain. |
| `OPACITY`, `CAP_OPACITY` | Higher is more opaque; lower more transparent (0–1). Transparency can complicate interpretation. |
| `SHOW_EDGES`, `SHOW_JUNCTION_LINES`, `SHOW_AUDIT_GRAIN`, `GMSH_SHOW_CURVES`, `SHOW_INTERNAL_INTERFACES`, `CAP_SHOW_EDGES`, `FINAL_SHOW_INTERNAL`, `PLOT_VALIDATION_ISSUES`, `SHOW_REPAIRED_SURFACES`, `SHOW_TET_CRINKLE_CLIP` | Display switches only: `True` shows the feature, `False` hides it. Gmsh curves can include computational seams. Crinkle clipping displays whole intersected tetrahedra. |
| `GMSH_VERBOSE` | Enables detailed Gmsh messages; no intended geometry change. |
| Names ending in `_OUTPUT` or `_AUDIT` | Saved-file locations, not numerical controls. |

**Pending work:** contact/overlap diagnosis, more rigorous voxel cleaning and the RVE coplanarity error await explicit **123go** authorization. These notes implement none of those fixes. Images alone cannot establish every shared-node contact or triangle intersection.


In [1]:
from pathlib import Path
import sys
import importlib
import numpy as np

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/upxo/meshing/gbconformant').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Start Jupyter inside the upxo_library checkout.')
sys.path.insert(0, str(ROOT / 'src'))
from upxo.meshing.gbconformant.d3v2p0.paths import data_directory
DATA_ROOT = data_directory()
RUN_DIR = DATA_ROOT / 'runs' / 'CM02'
RUN_DIR.mkdir(parents=True, exist_ok=True)
interface_module = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.interfaces'))
smooth_interfaces = interface_module.smooth_interfaces
SAMPLE = DATA_ROOT / 'samples/sblk_lgi.npy'
labels_original = np.load(SAMPLE, allow_pickle=False)

GRAIN_ID_SEED = 42
original_grain_ids = np.unique(labels_original)
shuffled_grain_ids = np.random.default_rng(GRAIN_ID_SEED).permutation(original_grain_ids)
grain_id_map = dict(zip(original_grain_ids.tolist(), shuffled_grain_ids.tolist()))
labels = shuffled_grain_ids[np.searchsorted(original_grain_ids, labels_original)]
print(f'{labels.shape}: {len(original_grain_ids)} grains')

(50, 50, 50): 373 grains


## Iterative UPXO spike removal before upscaling
Uses the existing `StructureCleaner3D._fast_clean_spikes` routine from `upxo.pxtal.twinned_simple_3d.cleaning_3d`. A spike has at most one face-connected neighbor with the same grain ID. Each pass reassigns spikes to neighboring grains while protecting the last voxel of every grain.

Set `CLEAN_SPIKES = False` to bypass cleaning, or adjust `SPIKE_MAX_PASSES`. Cleaning stops when a pass changes no voxels, the pass limit is reached, or a label configuration repeats. Remaining spike candidates can include protected singleton grains. Cleaning changes grain shapes and voxel counts; the original `labels` and input file remain intact. The optional pinch-cleaning step receives `labels_cleaned` before upscaling.


In [ ]:
CLEAN_SPIKES = True
SPIKE_MAX_PASSES = 100

# Always start from loaded/shuffled labels, not the result of a previous run.
labels_cleaned = labels.copy()
spike_history = []
spike_stop_reason = 'disabled'
if CLEAN_SPIKES:
    import hashlib
    from upxo.pxtal.twinned_simple_3d.cleaning_3d import StructureCleaner3D

    if isinstance(SPIKE_MAX_PASSES, bool) or not isinstance(SPIKE_MAX_PASSES, (int, np.integer)) or SPIKE_MAX_PASSES < 1:
        raise ValueError('SPIKE_MAX_PASSES must be a positive integer')
    if np.any(labels_cleaned < 0):
        raise ValueError('The UPXO spike cleaner requires nonnegative labels.')
    spike_cleaner = StructureCleaner3D(upscale_fallback=False, do_lobe_split=False)
    grain_ids_before_cleaning = np.unique(labels_cleaned)
    seen_states = {hashlib.sha256(labels_cleaned.tobytes()).digest()}
    spike_stop_reason = 'maximum passes reached'
    for pass_number in range(1, SPIKE_MAX_PASSES + 1):
        labels_cleaned, changed = spike_cleaner._fast_clean_spikes(labels_cleaned)
        spike_history.append({'pass': pass_number, 'changed_voxels': int(changed)})
        print(f'Spike pass {pass_number}: {changed} voxels reassigned')
        if changed == 0:
            spike_stop_reason = 'no further changes'
            break
        state = hashlib.sha256(labels_cleaned.tobytes()).digest()
        if state in seen_states:
            spike_stop_reason = 'repeated label configuration'
            break
        seen_states.add(state)
    assert np.array_equal(np.unique(labels_cleaned), grain_ids_before_cleaning), 'Cleaning lost a grain ID'
    remaining_candidates = np.count_nonzero(
        (labels_cleaned > 0) & (spike_cleaner._count_face_same_grain(labels_cleaned)
                               <= spike_cleaner._SPIKE_SAME_COUNT_THRESHOLD))
    print(f'Remaining spike candidates (including protected last voxels): {remaining_candidates}')
print('Spike cleaning stopped:', spike_stop_reason)
print('Net changed voxels:', np.count_nonzero(labels_cleaned != labels))
print('Grains retained:', len(np.unique(labels_cleaned)))


Spike pass 1: 950 voxels reassigned
Spike pass 2: 212 voxels reassigned
Spike pass 3: 37 voxels reassigned
Spike pass 4: 13 voxels reassigned
Spike pass 5: 7 voxels reassigned
Spike pass 6: 3 voxels reassigned
Spike pass 7: 2 voxels reassigned
Spike pass 8: 2 voxels reassigned
Remaining spike candidates (including protected last voxels): 195
Spike cleaning stopped: repeated label configuration
Net changed voxels: 1183
Grains retained: 373


## Optional targeted voxel pinch cleaning
Run after spike removal and before upscaling. `CLEAN_PINCHES` turns the step on or off. The detector searches 2 x 2 x 2 blocks for same-grain voxels connected only through edges/corners locally. It does not apply a majority filter. These are candidates, not proof that the contact is physically wrong; thin necks and pinches introduced later by smoothing are outside this detector's scope.

Each accepted one-voxel edit must remove candidates without introducing new ones, retain every grain ID, and preserve each grain's global face-connected component count. This does not guarantee preservation of tunnels, cavities, or all grain adjacency relationships. Small grains cannot donate voxels. Per-grain net volume change is limited to the greater of one voxel and the specified fraction of its initial count; a zero fraction forbids volume changes.

Controls cap passes, accepted edits, volume change and small-grain protection. Unresolved candidates remain reported. The input `labels_cleaned` remains unchanged; `labels_pinch_cleaned` feeds upscaling. The JSON audit records every voxel reassignment for inspection or replay. Compare on/off results with identical smoothing and Gmsh settings.


In [ ]:
CLEAN_PINCHES = True  # False bypasses this step completely.
PINCH_MAX_PASSES = 100
PINCH_MAX_CHANGES = 200
PINCH_MAX_VOLUME_FRACTION = 0.03
PINCH_PROTECT_GRAINS_UP_TO = 8  # Original voxel count, after spike cleaning.

import json
_pinch_module = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.pinch_cleaning'))
labels_pinch_cleaned, pinch_report, pinch_changes = _pinch_module.clean_pinch_contacts(
    labels_cleaned, enabled=CLEAN_PINCHES,
    max_passes=PINCH_MAX_PASSES, max_changes=PINCH_MAX_CHANGES,
    max_volume_fraction=PINCH_MAX_VOLUME_FRACTION,
    protect_grains_up_to=PINCH_PROTECT_GRAINS_UP_TO)
print(json.dumps(pinch_report, indent=2))
PINCH_AUDIT = RUN_DIR / 'CM02_pinch_cleaning.json'
PINCH_AUDIT.parent.mkdir(parents=True, exist_ok=True)
PINCH_AUDIT.write_text(json.dumps({'report': pinch_report, 'changes': pinch_changes}, indent=2))
print('Pinch-cleaning audit:', PINCH_AUDIT)


{
  "enabled": true,
  "initial_candidates": 482,
  "remaining_candidates": 261,
  "accepted_changes": 200,
  "net_changed_voxels": 200,
  "grains_retained": 373,
  "face_component_counts_preserved": true,
  "stop_reason": "maximum changes reached",
  "passes": [
    {
      "pass": 1,
      "accepted": 200,
      "remaining_candidates": 261
    }
  ],
  "grain_voxel_changes": {
    "3": 2,
    "4": 5,
    "6": -1,
    "15": -1,
    "17": 1,
    "23": -2,
    "24": 1,
    "34": -2,
    "35": -2,
    "45": -4,
    "46": 2,
    "51": 2,
    "52": 4,
    "53": 1,
    "72": 1,
    "78": -3,
    "81": -1,
    "86": 3,
    "91": -1,
    "106": -1,
    "110": 1,
    "112": -3,
    "114": -2,
    "116": 1,
    "122": 2,
    "124": 2,
    "129": -2,
    "131": 2,
    "138": -1,
    "139": 1,
    "143": -1,
    "184": 1,
    "185": 1,
    "190": -4,
    "198": 4,
    "200": 1,
    "204": -3,
    "205": 1,
    "207": -1,
    "208": 1,
    "209": 1,
    "211": -1,
    "213": -2,
    "221": -3,
   

## Optional resolution increase before smoothing
Set `UPSCALE_FACTOR` to an integer: `1` leaves the data unchanged, `2` doubles the resolution along each axis, and `3` triples it. Each original voxel is subdivided into identically labelled voxels using nearest-neighbor replication; no new grain IDs are introduced. Spacing decreases by the same factor, preserving physical RVE dimensions and grain shapes before smoothing.

For this sample, factor 2 changes 25 x 25 x 25 to 50 x 50 x 50. Voxel memory grows as factor cubed, and interface triangle count as factor squared. At higher resolution, the same Laplacian iteration count generally smooths over a smaller physical distance; increase `LAPLACE_ITERATIONS` if needed. Always refine from `labels_pinch_cleaned`, so rerunning this cell does not compound the upscaling.


In [4]:
UPSCALE_FACTOR = 2  # 1: unchanged; 2, 3, ...: subdivide along all three axes.
SPACING = (1.0, 1.0, 1.0)  # Original voxel spacing, before refinement.

# Refresh definitions if the notebook kernel predates this module update.
interface_module = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.interfaces'))
smooth_interfaces = interface_module.smooth_interfaces
labels_upscaled, effective_spacing = interface_module.upscale_labels(
    labels_pinch_cleaned, factor=UPSCALE_FACTOR, spacing=SPACING)
print(f'Voxel shape: {labels_pinch_cleaned.shape} -> {labels_upscaled.shape}')
print(f'Effective voxel spacing: {effective_spacing}')
print(f'Physical RVE dimensions: {np.array(labels_upscaled.shape) * effective_spacing}')


Voxel shape: (50, 50, 50) -> (100, 100, 100)
Effective voxel spacing: [0.5 0.5 0.5]
Physical RVE dimensions: [50. 50. 50.]


## Extract and smooth interfaces only
Change the iteration count and relaxation to control smoothing. Shared interfaces are extracted once per adjacent grain pair. Surface triangles are used for rendering; there is no volume meshing step.

Junction edges are identified from their incident grain interfaces, avoiding false branches caused by surface triangulation diagonals. Genuine junction points (four or more grains), branches and endpoints remain fixed. Movable degree-two junction nodes have independent controls:

- `JUNCTION_ITERATIONS`: line-smoothing steps; 0 disables line motion.
- `JUNCTION_RELAXATION`: smoothing rate; 0.25 is a damped starting value.
- `JUNCTION_MAX_DISPLACEMENT`: maximum total movement from the extracted coordinates, in physical units. The default uses 0.75 original voxel widths, independently of upscaling.

Surface smoothing and line smoothing advance together until their respective iteration limits. Triangle-collapse checks can restrict individual moves; frozen endpoints and branches can retain sharp turns. Increase surface iterations if surrounding surfaces constrain line motion. Turning-angle diagnostics refer only to movable degree-two nodes (0 degrees is straight).


In [5]:
LAPLACE_ITERATIONS = 50
LAPLACE_RELAXATION = 0.75

# Independent controls for actual junction-line edges.
JUNCTION_ITERATIONS = 10
JUNCTION_RELAXATION = 0.35  # Damped Laplacian step; 0 < value <= 1.
JUNCTION_MAX_DISPLACEMENT = 0.75 * min(SPACING)  # Physical units, from extracted geometry.

# Refresh the function even when only this cell is rerun after a source edit.
interface_module = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.interfaces'))
smooth_interfaces = interface_module.smooth_interfaces
interfaces = smooth_interfaces(labels_upscaled, spacing=effective_spacing,
                               iterations=LAPLACE_ITERATIONS,
                               relaxation=LAPLACE_RELAXATION,
                               junction_iterations=JUNCTION_ITERATIONS,
                               junction_relaxation=JUNCTION_RELAXATION,
                               junction_max_displacement=JUNCTION_MAX_DISPLACEMENT)
assert np.array_equal(interfaces.points[interfaces.node_kind == 3],
                      interfaces.original_points[interfaces.node_kind == 3])
assert np.array_equal(interfaces.points[interfaces.fixed_axes],
                      interfaces.original_points[interfaces.fixed_axes])
print(f'{len(interfaces.points):,} surface nodes; {len(interfaces.triangles):,} interface triangles')
print(f'Accepted Laplacian iterations: {interfaces.accepted_iterations}')
print('Frozen junction points:', np.count_nonzero(interfaces.node_kind == 3))
import json
print(json.dumps(interfaces.junction_metrics(), indent=2))

# Stable input for rerunning the optional point-contact separation cell.
interfaces_before_unpinch = interfaces


271,121 surface nodes; 557,896 interface triangles
Accepted Laplacian iterations: 50
Frozen junction points: 3196
{
  "movable_nodes": 20010,
  "junction_edges": 24086,
  "mean_turn_before_deg": 25.452773613193404,
  "mean_turn_after_deg": 11.743215484680293,
  "turns_over_45_before": 5659,
  "turns_over_45_after": 25,
  "maximum_line_displacement": 0.522876636288621
}


## Separate point contacts and press down their tips before Gmsh
This optional step intentionally removes even legitimate point contacts when the incident triangles form disconnected surface fans. Each fan receives its own node and its tip is relaxed inward with a smoothly decaying influence over the surrounding surface. Topology can change, and grain volumes are not preserved.

`SEPARATE_PINCHES` turns this on/off. Radius, maximum movement, and minimum opening are in physical units; the defaults scale with the original voxel spacing, independently of upscaling. Only the nodes at selected contacts are released from their former common-node/frozen constraints. Other frozen junction points remain fixed, and RVE coordinates stay on their original planes.

The step preserves edge incidence, guards against triangle collapse and per-step reversal, and rolls back the batch if any duplicated tips cannot be opened sufficiently. Connected surface fans and shared junction edges are not cut; separating those would require reconstructing interface patches. It does not certify the absence of global intersections or produce a tetrahedral mesh.

Rerun the smoothing cell once to populate `interfaces_before_unpinch`. After that, this cell can be rerun with different settings without compounding its previous changes. The surface plot and Gmsh cells below use the resulting `interfaces`.


In [6]:
SEPARATE_PINCHES = True
PINCH_PRESS_RADIUS = 1.5 * min(SPACING)
PINCH_PRESS_MAX_DISPLACEMENT = 0.5 * min(SPACING)
PINCH_PRESS_ITERATIONS = 30
PINCH_PRESS_RELAXATION = 0.25
PINCH_MIN_OPENING = 0.02 * min(SPACING)

import json
_unpinch_module = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.unpinch'))
interfaces, unpinch_report, unpinch_audit = _unpinch_module.separate_pinch_fans(
    interfaces_before_unpinch, enabled=SEPARATE_PINCHES,
    radius=PINCH_PRESS_RADIUS, max_displacement=PINCH_PRESS_MAX_DISPLACEMENT,
    iterations=PINCH_PRESS_ITERATIONS, relaxation=PINCH_PRESS_RELAXATION,
    min_separation=PINCH_MIN_OPENING)
print(json.dumps(unpinch_report, indent=2))
UNPINCH_AUDIT = RUN_DIR / 'CM02_unpinch.json'
UNPINCH_AUDIT.parent.mkdir(parents=True, exist_ok=True)
UNPINCH_AUDIT.write_text(json.dumps({'report': unpinch_report, 'contacts': unpinch_audit}, indent=2))
print('Point-contact repair audit:', UNPINCH_AUDIT)


{
  "enabled": true,
  "candidate_contacts": 130,
  "split_contacts": 130,
  "added_nodes": 130,
  "relaxation_steps": 30,
  "maximum_displacement": 0.500000000000003,
  "minimum_tip_separation": 0.6426475345767857,
  "unresolved_contacts": [],
  "stop_reason": "completed",
  "edge_incidence_preserved": true,
  "rve_planes_preserved": true
}
Point-contact repair audit: c:\Development\UPXO\upxo_library\src\upxo\meshing\confMesh3D_01\output\CM02_unpinch.json


## Plot only the smoothed internal grain boundary surfaces
Opens an interactive PyVista window. Rotate and zoom to inspect the interfaces.
Set `GRAIN_ID` to a shuffled grain ID to isolate its internal interfaces, or leave it as `None` to show all interfaces. Each shared interface is coloured by its grain pair. No outer caps, tetrahedra or mesh edges are displayed by default.

In [7]:
import pyvista as pv

GRAIN_ID = None  # Example: int(np.unique(labels)[0]); IDs refer to shuffled labels.
SHOW_EDGES = False
SHOW_JUNCTION_LINES = False  # Overlay the actual junction graph.
OPACITY = 1.0

selected = (np.ones(len(interfaces.triangles), dtype=bool) if GRAIN_ID is None
            else np.any(interfaces.grain_pairs == GRAIN_ID, axis=1))
if not np.any(selected):
    print('No internal interfaces for this selection.')
else:
    triangles = interfaces.triangles[selected]
    faces = np.column_stack((np.full(len(triangles), 3), triangles)).ravel()
    boundary_surfaces = pv.PolyData(interfaces.points, faces)
    pairs = np.sort(interfaces.grain_pairs, axis=1)
    unique_pairs, pair_ids = np.unique(pairs, axis=0, return_inverse=True)
    pair_colours = np.random.default_rng(GRAIN_ID_SEED).permutation(len(unique_pairs))
    boundary_surfaces.cell_data['interface_colour'] = pair_colours[pair_ids[selected]]
    boundary_surfaces.cell_data['grain_a'] = pairs[selected, 0]
    boundary_surfaces.cell_data['grain_b'] = pairs[selected, 1]

    surface_plotter = pv.Plotter(notebook=False)
    surface_plotter.add_mesh(boundary_surfaces, scalars='interface_colour',
                             cmap='nipy_spectral', show_scalar_bar=False,
                             show_edges=SHOW_EDGES, opacity=OPACITY)
    if SHOW_JUNCTION_LINES and len(interfaces.junction_edges):
        line_edges = interfaces.junction_edges
        if GRAIN_ID is not None:
            visible_nodes = np.unique(triangles)
            line_edges = line_edges[np.all(np.isin(line_edges, visible_nodes), axis=1)]
        if len(line_edges):
            junction_lines = pv.PolyData(interfaces.points)
            junction_lines.lines = np.column_stack((np.full(len(line_edges), 2), line_edges)).ravel()
            surface_plotter.add_mesh(junction_lines, color='black', line_width=3)
    surface_plotter.add_text('Laplacian-smoothed grain interfaces', font_size=12)
    surface_plotter.add_axes()
    surface_plotter.show()


### Trace a small rounded feature to its voxel grain
No circular healing patches or point markers are inserted by the interface plot. Small retained grain regions can become rounded during smoothing, and contact separation can change their visibility. This audit traces a chosen original grain ID through colour shuffling and cleaning to its actual surface triangles. It does not remove small grains or assert that a rounded rendering is a physically spherical sub-block.


In [8]:
AUDIT_ORIGINAL_GRAIN_ID = None  # None selects a grain with the fewest input voxels.
SHOW_AUDIT_GRAIN = False

audit_ids, audit_counts = np.unique(labels_original, return_counts=True)
print('Input grain IDs occupying one voxel:', int(np.sum(audit_counts == 1)))
print('Input grain IDs occupying at most eight voxels:', int(np.sum(audit_counts <= 8)))
audit_original_id = (int(audit_ids[np.argmin(audit_counts)])
                     if AUDIT_ORIGINAL_GRAIN_ID is None else int(AUDIT_ORIGINAL_GRAIN_ID))
audit_display_id = grain_id_map[audit_original_id]
audit_triangles = interfaces.triangles[np.any(interfaces.grain_pairs == audit_display_id, axis=1)]
print('Original ID:', audit_original_id, '| shuffled/display ID:', audit_display_id)
print('Original voxels:', int(np.count_nonzero(labels_original == audit_original_id)))
print('Voxels after cleaning (before upscaling):',
      int(np.count_nonzero(labels_pinch_cleaned == audit_display_id)))
print('Internal interface triangles:', len(audit_triangles))
if SHOW_AUDIT_GRAIN and len(audit_triangles):
    audit_nodes, audit_conn = np.unique(audit_triangles, return_inverse=True)
    audit_surface = pv.PolyData(interfaces.points[audit_nodes],
        np.column_stack((np.full(len(audit_triangles), 3), audit_conn.reshape(-1, 3))).ravel())
    audit_plotter = pv.Plotter(notebook=False)
    audit_plotter.add_mesh(audit_surface, color='orange', show_edges=True)
    audit_plotter.add_text(f'Original grain {audit_original_id} | display ID {audit_display_id}', font_size=11)
    audit_plotter.add_axes()
    audit_plotter.show()


Input grain IDs occupying one voxel: 56
Input grain IDs occupying at most eight voxels: 185
Original ID: 14 | shuffled/display ID: 165
Original voxels: 1
Voxels after cleaning (before upscaling): 1
Internal interface triangles: 48


## Gmsh remeshing with connected interface boundaries
Gmsh builds shared discrete curves from the labelled surface topology, parametrizes each surface, and regenerates its 2D triangles. Existing curve nodes and edges are retained: neighboring interfaces therefore share the same boundary discretization, and traces on RVE faces stay on those faces at their smoothed positions. Frozen junction points also remain fixed. No volume mesh or exterior caps are generated.

`GMSH_SIZE` is the target interior triangle size in physical units; fixed boundary segments retain their existing lengths. Parametrization can split a grain-pair interface into several charts. The step checks every chart boundary and preserves all grain-pair labels; failures raise an error instead of silently skipping an interface. The original `interfaces` object is unchanged.

Uses Gmsh's [discrete geometry and topology APIs](https://gmsh.info/doc/texinfo/gmsh.html#Namespace-gmsh_002fmodel_002fmesh), with `Mesh.MeshOnlyEmpty` to retain the curve mesh. Requires `gmsh` and `pyvista` in the selected kernel (verified with Gmsh 4.14.1). The VTP export retains both grain IDs for every triangle.


Change `GMSH_ALGORITHM` to compare triangular surface meshers: 1 MeshAdapt, 2 Automatic, 5 Delaunay, 6 Frontal-Delaunay (the previous default). Keep geometry and `GMSH_SIZE` fixed when comparing. Gmsh can fall back internally to MeshAdapt; use `GMSH_VERBOSE = True` to inspect its messages. The report records the requested algorithm, not a guarantee of which fallback ran. Shared-curve and RVE constraints remain checked for every option. Changing the algorithm does not repair a distorted underlying parametrization.


In [9]:
# Regenerate all grain-interface surface triangles in Gmsh.
import importlib
import json
import pyvista as pv

GMSH_SIZE = 0.75 * min(SPACING)  # Physical target size; boundary curves stay fixed.
GMSH_ALGORITHM = 6  # 1: MeshAdapt; 2: Automatic; 5: Delaunay; 6: Frontal-Delaunay
GMSH_VERBOSE = False
GMSH_SHOW_CURVES = False

_gmsh_remesher = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.gmsh_interfaces'))
gmsh_result = _gmsh_remesher.remesh_interfaces_gmsh(
    interfaces, mesh_size=GMSH_SIZE, verbose=GMSH_VERBOSE,
    algorithm=GMSH_ALGORITHM)
print(json.dumps(gmsh_result.report, indent=2))

cells = np.column_stack((np.full(len(gmsh_result.triangles), 3), gmsh_result.triangles)).ravel()
gmsh_surfaces = pv.PolyData(gmsh_result.points, cells)
gmsh_surfaces.cell_data['grain_a'] = gmsh_result.grain_pairs[:, 0]
gmsh_surfaces.cell_data['grain_b'] = gmsh_result.grain_pairs[:, 1]
remeshed_pairs, remeshed_pair_ids = np.unique(gmsh_result.grain_pairs, axis=0, return_inverse=True)
pair_colours = np.random.default_rng(GRAIN_ID_SEED).permutation(len(remeshed_pairs))
gmsh_surfaces.cell_data['interface_colour'] = pair_colours[remeshed_pair_ids]

GMSH_OUTPUT = RUN_DIR / 'CM02_gmsh.vtp'
GMSH_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
gmsh_surfaces.save(GMSH_OUTPUT)
GMSH_OUTPUT.with_suffix('.json').write_text(json.dumps(gmsh_result.report, indent=2))
print('Saved:', GMSH_OUTPUT)

{
  "input_triangles": 557896,
  "output_triangles": 738974,
  "requested_algorithm": 6,
  "source_chart_orientation_verified": true,
  "reoriented_surface_charts": [
    407,
    724,
    1330
  ],
  "grain_pairs": 1607,
  "surface_charts": 3089,
  "preserved_curve_nodes": 43201,
  "shared_boundaries_verified": true,
  "curve_coordinates_preserved": true,
  "mesh_size": 0.75,
  "rve_trace_nodes_preserved": 10058,
  "frozen_junction_points_preserved": 3326,
  "surface_meshing_warnings": [],
  "surface_warning_capture_available": true,
  "maximum_curve_coordinate_error": 0.0,
  "parametrization_repairs": [
    {
      "failed_charts": 30,
      "replacement_disk_charts": 1225,
      "source_triangles": 44272,
      "errors": [
        "Source chart is not a manifold disk",
        "Source chart is not a manifold disk",
        "Source chart is not a manifold disk",
        "Source chart is not a manifold disk",
        "Source chart is not a manifold disk"
      ]
    },
    {
      "st

In [10]:
gmsh_plotter = pv.Plotter(notebook=False)
gmsh_plotter.add_mesh(gmsh_surfaces, scalars='interface_colour', cmap='nipy_spectral',
                     show_edges=False, edge_color='gray', show_scalar_bar=False)
if GMSH_SHOW_CURVES and len(gmsh_result.curve_edges):
    preserved_curves = pv.PolyData(gmsh_result.points)
    preserved_curves.lines = np.column_stack((np.full(len(gmsh_result.curve_edges), 2),
                                              gmsh_result.curve_edges)).ravel()
    gmsh_plotter.add_mesh(preserved_curves, color='black', line_width=2)
gmsh_plotter.add_text('Gmsh-remeshed interfaces | shared curves fixed', font_size=11)
gmsh_plotter.add_axes()
gmsh_plotter.show()


## Close all six RVE faces
Build grain-labelled planar caps from the current Gmsh interface traces and the RVE perimeter. Boundary segments reuse the existing interface node IDs without moving them or subdividing their edges. Caps are triangulated by Gmsh and joined into one surface complex.

This cell checks complete face coverage, a closed exterior shell, per-grain edge closure, positive enclosed signed grain volumes and total RVE volume. Existing nonmanifold grain edges are reported separately; capping does not repair those or certify absence of self-intersections. No tetrahedra are generated here.

Run the preceding Gmsh cell first. `CAP_MESH_SIZE` controls cap-interior resolution; boundary segments remain fixed. `SHOW_INTERNAL_INTERFACES` can overlay the internal surfaces, and reducing `CAP_OPACITY` reveals them. The complete closed surface complex, the exterior caps alone, and a validation report are exported. On cap triangles `grain_a == grain_b`; `is_rve_cap` distinguishes them from internal interfaces. Face IDs 0..5 mean x-min, x-max, y-min, y-max, z-min, z-max.

Box perimeter edges are subdivided using `RVE_EDGE_SIZE`; adjacent RVE faces share these nodes. Rerun this closure cell, then the final joint remeshing cell after changing the size.


In [11]:
import importlib
import json
import pyvista as pv

CAP_MESH_SIZE = GMSH_SIZE
RVE_EDGE_SIZE = CAP_MESH_SIZE  # Maximum new box-edge segment length
CAP_OPACITY = 1.0
SHOW_INTERNAL_INTERFACES = False
CAP_SHOW_EDGES = True

_rve_caps = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.rve_caps'))
closed_rve = _rve_caps.close_rve_faces(
    gmsh_result, labels_upscaled, spacing=effective_spacing,
    mesh_size=CAP_MESH_SIZE, edge_size=RVE_EDGE_SIZE)
print(json.dumps({k: v for k, v in closed_rve.report.items()
                  if k != 'enclosed_grain_volumes'}, indent=2))

closed_cells = np.column_stack((np.full(len(closed_rve.triangles), 3),
                                closed_rve.triangles)).ravel()
closed_rve_surface = pv.PolyData(closed_rve.points, closed_cells)
closed_rve_surface.cell_data['grain_a'] = closed_rve.grain_pairs[:, 0]
closed_rve_surface.cell_data['grain_b'] = closed_rve.grain_pairs[:, 1]
closed_rve_surface.cell_data['is_rve_cap'] = closed_rve.exterior.astype(np.uint8)
closed_rve_surface.cell_data['rve_face'] = closed_rve.rve_face
rve_caps_surface = closed_rve_surface.extract_cells(closed_rve.exterior).extract_surface()

CLOSED_RVE_OUTPUT = RUN_DIR / 'CM02_closed_RVE.vtp'
CLOSED_RVE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
closed_rve_surface.save(CLOSED_RVE_OUTPUT)
rve_caps_surface.save(CLOSED_RVE_OUTPUT.with_name('CM02_RVE_caps.vtp'))
CLOSED_RVE_OUTPUT.with_suffix('.json').write_text(json.dumps(closed_rve.report, indent=2))
print('Saved closed surface complex:', CLOSED_RVE_OUTPUT)

rve_plotter = pv.Plotter(notebook=False)
rve_plotter.add_mesh(rve_caps_surface, scalars='grain_a', cmap='nipy_spectral',
                     show_edges=CAP_SHOW_EDGES, edge_color='gray', opacity=CAP_OPACITY,
                     show_scalar_bar=False)
if SHOW_INTERNAL_INTERFACES:
    rve_plotter.add_mesh(gmsh_surfaces, scalars='grain_a', cmap='nipy_spectral',
                         show_edges=False, show_scalar_bar=False)
rve_plotter.add_text('Closed RVE faces | grain-labelled caps', font_size=12)
rve_plotter.add_axes()
rve_plotter.show()


ValueError: RVE face 1: an internal triangle is coplanar with the cap

### Joint Gmsh remeshing: RVE caps and internal interfaces
Run after the RVE closure cell. All labelled surface patches are regenerated together. Shared curve nodes/edges stay fixed; cap interior nodes are remeshed on their RVE planes. This is surface meshing only. The report checks closure and lists any nonmanifold grain edges that still require repair before tetrahedral meshing.


In [ ]:
# Remesh all caps and internal interfaces in one Gmsh model.
FINAL_GMSH_SIZE = GMSH_SIZE
FINAL_GMSH_ALGORITHM = GMSH_ALGORITHM  # 1, 2, 5, or 6
FINAL_SHOW_INTERNAL = False

from importlib import import_module, reload
reload(import_module('upxo.meshing.gbconformant.d3v2p0.gmsh_interfaces'))
_joint = reload(import_module('upxo.meshing.gbconformant.d3v2p0.gmsh_closed'))
remeshed_rve = _joint.remesh_closed_rve_gmsh(
    closed_rve, mesh_size=FINAL_GMSH_SIZE, algorithm=FINAL_GMSH_ALGORITHM)
print({k: v for k, v in remeshed_rve.report.items()
       if k != 'enclosed_grain_volumes'})

remeshed_rve_surface = pv.PolyData(
    remeshed_rve.points,
    np.column_stack((np.full(len(remeshed_rve.triangles), 3),
                     remeshed_rve.triangles)).ravel())
for key, values in dict(grain_a=remeshed_rve.grain_pairs[:, 0],
                        grain_b=remeshed_rve.grain_pairs[:, 1],
                        is_rve_cap=remeshed_rve.exterior.astype(np.uint8),
                        rve_face=remeshed_rve.rve_face).items():
    remeshed_rve_surface.cell_data[key] = values
remeshed_rve_caps = remeshed_rve_surface.extract_cells(
    remeshed_rve.exterior).extract_surface()
FINAL_RVE_OUTPUT = CLOSED_RVE_OUTPUT.with_name('CM02_closed_RVE_remeshed.vtp')
remeshed_rve_surface.save(FINAL_RVE_OUTPUT)
import json
FINAL_RVE_OUTPUT.with_suffix('.json').write_text(
    json.dumps(remeshed_rve.report, indent=2))
final_plotter = pv.Plotter(notebook=False)
final_plotter.add_mesh(remeshed_rve_caps, scalars='grain_a', cmap='nipy_spectral',
                       show_edges=True, opacity=0.35 if FINAL_SHOW_INTERNAL else 1.)
if FINAL_SHOW_INTERNAL:
    final_plotter.add_mesh(remeshed_rve_surface.extract_cells(~remeshed_rve.exterior),
                           scalars='grain_a', cmap='nipy_spectral', show_edges=True)
final_plotter.add_axes()
final_plotter.show()


### Pre-tetrahedral-meshing surface validation
Run after the final joint surface remeshing cell. This read-only check reports per-grain closure, edge/vertex manifoldness, orientation, ownership, duplicate triangles/nodes, triangle quality, RVE planes/areas and signed volumes. Shared triple lines are checked per grain. `BLOCKED` means defects were found. A preliminary pass is **not** a guarantee of successful tetrahedral meshing: triangle intersections, clearance and shell containment remain unchecked. No tetrahedra are generated.


In [ ]:
# Validate the final surface complex before attempting Gmsh volume meshing.
MIN_SURFACE_TRIANGLE_QUALITY = 0.05
PLOT_VALIDATION_ISSUES = True
STOP_ON_VALIDATION_ERRORS = False  # True raises after saving the report.

from importlib import import_module, reload
import json
_validation = reload(import_module('upxo.meshing.gbconformant.d3v2p0.tet_validation'))
tet_validation = _validation.validate_tet_surfaces(
    remeshed_rve, expected_grains=np.unique(labels_upscaled),
    min_triangle_quality=MIN_SURFACE_TRIANGLE_QUALITY)
print('Pre-tet validation:', tet_validation['status'])
print('Triangle quality:', tet_validation.get('triangle_quality'))
for category in ('blockers', 'warnings', 'unchecked'):
    print('\n' + category.upper())
    for message in tet_validation[category]:
        print(' -', message)
print('\nPer-grain defects:')
for grain, checks in tet_validation['per_grain'].items():
    if any(checks[k] for k in ('open_edges', 'nonmanifold_edges',
                               'inconsistent_oriented_edges', 'nonmanifold_vertices')):
        print(grain, checks)
VALIDATION_OUTPUT = FINAL_RVE_OUTPUT.with_name('CM02_pre_tet_validation.json')
VALIDATION_OUTPUT.write_text(json.dumps(tet_validation, indent=2))
print('Saved:', VALIDATION_OUTPUT)

if PLOT_VALIDATION_ISSUES and tet_validation['bad_triangle_ids']:
    issue_plotter = pv.Plotter(notebook=False)
    issue_plotter.add_mesh(remeshed_rve_surface, color='lightgray', opacity=0.1)
    issue_plotter.add_mesh(remeshed_rve_surface.extract_cells(
        np.asarray(tet_validation['bad_triangle_ids'], dtype=int)),
        color='red', show_edges=True)
    issue_plotter.add_text('Red: topology/orientation defects or poor triangles', font_size=10)
    issue_plotter.add_axes()
    issue_plotter.show()
if STOP_ON_VALIDATION_ERRORS and tet_validation['blockers']:
    raise RuntimeError('Surface validation blocked: inspect tet_validation and its JSON report.')


### Repair contacts, revalidate, and verify tetrahedral meshing
These cells use the final `remeshed_rve` without altering it. The repair separates disconnected sheets at point/edge contacts and moves only those contact nodes by a bounded amount. True shared junctions remain connected and box-plane coordinates remain fixed. This deliberately changes local grain morphology. Unsupported defects stop the subsequent volume-meshing cell.

The final cell generates a real Gmsh tetrahedral mesh, including disconnected pieces with their original grain IDs. It checks positive element volumes, every grain's surface conformity and volume, and minimum tetrahedron quality. `TET_MESH_VERIFIED` applies to the generated mesh and selected quality threshold; it is not a universal guarantee for different settings or future datasets.

The intersection-repair cell separates crossing interface wedges while freezing their shared junctions. Final validation and the tetrahedral mesher both test geometric triangle intersections, including crossings between triangles that share a node. These are tolerance-based checks; minimum-clearance certification remains separate.


In [ ]:
from importlib import import_module, reload
import json

REPAIR_CONTACTS = True
REPAIR_SEPARATION = 0.15 * float(np.min(SPACING))  # Maximum movement in physical units
SHOW_REPAIRED_SURFACES = True

_repair = reload(import_module('upxo.meshing.gbconformant.d3v2p0.tet_repair'))
repaired_rve = (_repair.repair_tet_contacts(remeshed_rve, separation=REPAIR_SEPARATION)
                if REPAIR_CONTACTS else remeshed_rve)
print('Contact repair:', repaired_rve.report.get('contact_repair', 'disabled'))
repaired_rve_surface = pv.PolyData(
    repaired_rve.points,
    np.column_stack((np.full(len(repaired_rve.triangles), 3), repaired_rve.triangles)).ravel())
repaired_rve_surface.cell_data['grain_a'] = repaired_rve.grain_pairs[:, 0]
repaired_rve_surface.cell_data['grain_b'] = repaired_rve.grain_pairs[:, 1]
repaired_rve_surface.cell_data['is_rve_cap'] = repaired_rve.exterior.astype(np.uint8)
repaired_rve_surface.cell_data['rve_face'] = repaired_rve.rve_face
REPAIRED_OUTPUT = FINAL_RVE_OUTPUT.with_name('CM02_repaired_RVE.vtp')
repaired_rve_surface.save(REPAIRED_OUTPUT)
if SHOW_REPAIRED_SURFACES:
    repair_plotter = pv.Plotter(notebook=False)
    repair_plotter.add_mesh(repaired_rve_surface, scalars='grain_a', cmap='nipy_spectral',
                             show_edges=True, opacity=0.5)
    moved = repaired_rve.report.get('contact_repair', {}).get('split_source_nodes', [])
    if moved:
        repair_plotter.add_points(remeshed_rve.points[moved], color='red',
                                  point_size=12, render_points_as_spheres=True)
    repair_plotter.add_axes()
    repair_plotter.show()


In [ ]:
# Remove geometric crossings beside junction lines before the final checks.
REPAIR_INTERFACE_CROSSINGS = True
CROSSING_MAX_DISPLACEMENT = 0.35 * float(np.min(SPACING))
CROSSING_NEIGHBOURHOOD_RADIUS = 1.0 * float(np.min(SPACING))
CROSSING_OPENING = 0.10  # Separation relative to distance from the junction line

_crossings = reload(import_module('upxo.meshing.gbconformant.d3v2p0.intersection_repair'))
if REPAIR_INTERFACE_CROSSINGS:
    repaired_rve = _crossings.repair_interface_crossings(
        repaired_rve, max_displacement=CROSSING_MAX_DISPLACEMENT,
        radius=CROSSING_NEIGHBOURHOOD_RADIUS, opening=CROSSING_OPENING)
    print('Intersection repair:', repaired_rve.report['intersection_repair'])
    repaired_rve_surface.points = repaired_rve.points
    repaired_rve_surface.save(REPAIRED_OUTPUT)
else:
    print('Intersection repair disabled; the following checks still detect crossings.')


In [ ]:
# A fresh report is required: the pre-repair report is not reused.
_validation = reload(import_module('upxo.meshing.gbconformant.d3v2p0.tet_validation'))
repaired_validation = _validation.validate_tet_surfaces(
    repaired_rve, expected_grains=np.unique(labels_upscaled),
    min_triangle_quality=MIN_SURFACE_TRIANGLE_QUALITY, check_intersections=True)
print('After repair:', repaired_validation['status'])
for category in ('blockers', 'warnings', 'unchecked'):
    print(category.upper(), repaired_validation[category])
print('Surface quality:', repaired_validation.get('triangle_quality'))
REPAIRED_OUTPUT.with_suffix('.json').write_text(json.dumps({
    'repair': repaired_rve.report.get('contact_repair'),
    'intersection_repair': repaired_rve.report.get('intersection_repair'),
    'validation': repaired_validation}, indent=2))
if repaired_validation['blockers']:
    raise RuntimeError('Repair has not resolved all surface defects; inspect repaired_validation.')


In [ ]:
# Generate and verify tetrahedra; readiness is based on this actual result.
RUN_TET_VERIFICATION = True
TET_MESH_SIZE = 1.5 * float(np.min(SPACING))
TET_OPTIMIZE_NETGEN = True  # Keep baseline if optimization worsens a volume
MIN_TET_QUALITY = 0.05  # Gmsh minSICN: larger positive values are better
SHOW_TET_CRINKLE_CLIP = True

tet_mesh_ready = False
verified_tets = None
if RUN_TET_VERIFICATION:
    reload(import_module('upxo.meshing.gbconformant.d3v2p0.tet_validation'))
    _tets = reload(import_module('upxo.meshing.gbconformant.d3v2p0.gmsh_tets'))
    # Rechecked inside the mesher, even if notebook cells were run out of order.
    verified_tets = _tets.mesh_repaired_rve_gmsh(
        repaired_rve, mesh_size=TET_MESH_SIZE, optimize_netgen=TET_OPTIMIZE_NETGEN,
        minimum_quality=MIN_TET_QUALITY)
    tet_mesh_ready = verified_tets.report['ready']
    print(json.dumps(verified_tets.report, indent=2))
    tet_grid = pv.UnstructuredGrid(
        np.column_stack((np.full(len(verified_tets.tetrahedra), 4),
                         verified_tets.tetrahedra)).ravel(),
        np.full(len(verified_tets.tetrahedra), pv.CellType.TETRA, dtype=np.uint8),
        verified_tets.points)
    tet_grid.cell_data['grain_id'] = verified_tets.grain_ids
    tet_grid.cell_data['minSICN'] = verified_tets.quality
    TET_OUTPUT = FINAL_RVE_OUTPUT.with_name('CM02_verified_tets.vtu')
    tet_grid.save(TET_OUTPUT)
    TET_OUTPUT.with_suffix('.json').write_text(json.dumps(verified_tets.report, indent=2))
    if SHOW_TET_CRINKLE_CLIP:
        tet_plotter = pv.Plotter(notebook=False)
        tet_plotter.add_mesh_clip_plane(tet_grid, scalars='grain_id', cmap='tab20c',
                                        show_edges=True, crinkle=True, normal='x')
        tet_plotter.add_axes()
        tet_plotter.show()
    if not tet_mesh_ready:
        print('Tetrahedra generated, but the requested quality threshold was not met.')
else:
    print('Tet verification skipped; tet_mesh_ready remains False.')


In [ ]:
tet_plotter = pv.Plotter(notebook=False)
tet_plotter.add_mesh_clip_plane(tet_grid, scalars='grain_id', cmap='tab20c',
                                show_edges=False, crinkle=True, normal='x')
tet_plotter.add_axes()
tet_plotter.show()